# Optimization loops

This tutorial explains how optimization works in `f3dasm`. Optimization in `f3dasm` is expressed by composing **Blocks** — there is no separate `Optimizer` base class. The key building blocks are:

- The `>>` operator for chaining blocks into a `ChainedBlock`.
- The `LoopBlock` (or the `.loop(n)` convenience method on any block) for repeating a block `n` times.
- Two shapes of optimizer blocks: **ask/tell update-step blocks** (e.g. TPE) and **one-shot blocks** (e.g. scipy's `minimize`).

By the end you will know when to use each shape and how to write your own.

## 1. Blocks recap

Every computational unit in `f3dasm` is a `Block` with the signature:

```python
class Block(ABC):
    def arm(self, data: ExperimentData) -> None: ...       # optional one-time setup
    def call(self, data: ExperimentData, **kwargs) -> ExperimentData: ...
```

Samplers, data generators, and optimizer update steps are all `Block` subclasses. They compose in two ways:

### The `>>` operator

`block_a >> block_b` returns a `ChainedBlock` that runs `block_a`, feeds its output into `block_b`, and returns `block_b`'s result. Chaining is associative and any number of blocks can be joined:

```python
pipeline = block_a >> block_b >> block_c
data = pipeline.call(data)
```

### `LoopBlock` and `.loop(n)`

`LoopBlock(block, n_iterations=n)` runs an inner block `n` times, feeding each iteration's output as the next iteration's input. Any block has a `.loop(n)` convenience method:

```python
loop = my_block.loop(50)                       # LoopBlock wrapping my_block
loop = (step >> data_gen).loop(50)             # also works on ChainedBlock
```

This is the primitive that drives optimization loops.

## 2. The canonical optimization pattern

Most optimization algorithms in `f3dasm` follow an **ask/tell** cycle:

1. The optimizer *asks* for a new candidate based on what it has seen so far.
2. A data generator *evaluates* the candidate.
3. The optimizer *tells* the result back into its internal state on the next iteration.

In `f3dasm` this is expressed as:

```python
loop = (update_step >> data_generator).loop(n_iterations)
loop.arm(data)
data = loop.call(data)
```

where `update_step` is an optimizer block that returns `data` with a single new **unevaluated** row appended, and `data_generator` evaluates any open jobs. Each iteration therefore appends one new evaluated row to `data`.

## 3. A concrete example: TPE on a 2D objective

We minimize a simple paraboloid $(x_0 - 1.5)^2 + (x_1 + 2)^2$ using the built-in Tree-structured Parzen Estimator (TPE) sampler from Optuna as the update step.

In [ ]:
from f3dasm import ExperimentData, create_sampler, datagenerator
from f3dasm.design import Domain
from f3dasm.optimization import tpesampler


# 1. Domain: two continuous inputs plus one output
domain = Domain()
domain.add_float(name="x0", low=-5.0, high=5.0)
domain.add_float(name="x1", low=-5.0, high=5.0)
domain.add_output("y")


# 2. Data generator: a plain Python function wrapped with @datagenerator
@datagenerator(output_names="y")
def f(x0: float, x1: float) -> float:
    return (x0 - 1.5) ** 2 + (x1 + 2.0) ** 2


# 3. Initial design: sample with Latin Hypercube, then evaluate
data = ExperimentData(domain=domain)
data = create_sampler("latin", seed=42).call(data, n_samples=10)

f.arm(data)
data = f.call(data)

# 4. Optimize: (update_step >> data_generator).loop(n)
update_step = tpesampler(output_name="y")
loop = (update_step >> f).loop(50)
loop.arm(data)
data = loop.call(data)

print(
    f"Total rows after optimization: {len(data)}"
)  # 10 initial + 50 iterations

Notice that we:

- **Separate sampling from evaluation** in the initial design step. This is because the sampler's `call` takes an `n_samples` argument that the data generator doesn't accept; chaining them with `>>` would forward `n_samples` into `f` and fail. Keep incompatible call signatures in separate `.call()` invocations.
- **Compose the optimizer update step with the data generator** via `>>`. Both accept only `**kwargs` that they can safely ignore, so chaining is clean.
- **Wrap the chain in `.loop(n_iterations)`**. Without the loop, `(update_step >> f).call(data)` would only perform one iteration.

In [ ]:
# Inspect the best result
df_in, df_out = data.to_pandas()
best_idx = df_out["y"].idxmin()
print(
    f"Best: x0={df_in.loc[best_idx, 'x0']:.3f}, "
    f"x1={df_in.loc[best_idx, 'x1']:.3f}, "
    f"y={df_out.loc[best_idx, 'y']:.4f}"
)

## 4. The `create_optimizer` factory

`f3dasm.create_optimizer(name, **kwargs)` is a string-based factory that returns an optimizer block. For ask/tell optimizers it returns *just the update step*, so the caller is still responsible for chaining with a data generator and wrapping in a `LoopBlock`:

```python
from f3dasm import create_optimizer

update_step = create_optimizer("tpesampler", output_name="y")
loop = (update_step >> f).loop(50)
```

Equivalent to importing `tpesampler` directly from `f3dasm.optimization`.

## 5. One-shot optimizers (scipy)

Not every optimizer fits the ask/tell shape. The scipy optimizers (`cg`, `lbfgsb`, `nelder_mead`) run their own inner loop inside `scipy.optimize.minimize` — `f3dasm` can't intervene between iterations. For these, `f3dasm` exposes a **one-shot block**:

- The block's `call()` runs scipy's full inner loop on a single invocation.
- Pass the iteration budget as `maxiter` in the factory's keyword arguments.
- **Do not wrap it in `LoopBlock`** — that would run scipy's full loop `n` times.

Scipy's `minimize` calls the objective as `f(x)` where `x` is a flat numpy array, so the domain must have a single `ArrayParameter` named the same as `input_name`:

In [ ]:
import numpy as np

from f3dasm.optimization import lbfgsb

# Scipy wants a single array-valued input
scipy_domain = Domain()
scipy_domain.add_array(name="x", shape=(2,), low=-5.0, high=5.0)
scipy_domain.add_output("y")


@datagenerator(output_names="y")
def f_array(x: np.ndarray) -> float:
    return float(np.sum((x - np.array([1.5, -2.0])) ** 2))


scipy_data = ExperimentData(domain=scipy_domain)
scipy_data = create_sampler("random", seed=0).call(scipy_data, n_samples=1)
f_array.arm(scipy_data)
scipy_data = f_array.call(scipy_data)

scipy_opt = lbfgsb(
    data_generator=f_array,
    output_name="y",
    input_name="x",
    maxiter=50,
)
scipy_opt.arm(scipy_data)
scipy_data = scipy_opt.call(scipy_data)  # single call runs scipy's full loop

print(f"Rows after scipy's inner loop: {len(scipy_data)}")

This is a deliberate asymmetry: scipy's API doesn't expose per-iteration state the way ask/tell samplers do, so we can't fit it into the same `LoopBlock` pattern. The same factory surface still applies: `create_optimizer("lbfgsb", data_generator=..., output_name=..., input_name=..., maxiter=50)`.

## 6. Writing your own update-step block

Any ask/tell algorithm can become an f3dasm optimizer by subclassing `Block` and implementing two methods. The contract for a single `call`:

1. Register the previous iteration's result with your internal model (skip on the first call).
2. Ask your algorithm for one new candidate.
3. Append it as an **unevaluated** row to `data` and return the result.

`arm(data)` should seed your internal model with whatever evaluated rows `data` already contains.

In [ ]:
from f3dasm import Block
from f3dasm._src.experimentsample import ExperimentSample


class RandomSearchUpdateStep(Block):
    """Toy ask/tell optimizer that proposes uniform-random candidates.

    This is deliberately naive; a real optimizer would use the history
    of input/output pairs to inform its suggestion.
    """

    def __init__(self, seed: int | None = None) -> None:
        self._rng = np.random.default_rng(seed)

    def arm(self, data: ExperimentData) -> None:
        # No model state to seed in this toy example.
        pass

    def call(self, data: ExperimentData, **kwargs) -> ExperimentData:
        sample_values = {}
        for name, parameter in data.domain.input_space.items():
            sample_values[name] = float(
                self._rng.uniform(
                    low=parameter.lower_bound, high=parameter.upper_bound
                )
            )

        new_es = ExperimentSample(_input_data=sample_values)
        new_data = ExperimentData.from_data(
            data={0: new_es},
            domain=data.domain,
            project_dir=data._project_dir,
        )
        return data + new_data


# Use it in the canonical pattern
rs_data = ExperimentData(domain=domain)
rs_data = create_sampler("latin", seed=42).call(rs_data, n_samples=5)
f.arm(rs_data)
rs_data = f.call(rs_data)

loop = (RandomSearchUpdateStep(seed=0) >> f).loop(20)
loop.arm(rs_data)
rs_data = loop.call(rs_data)

print(f"Rows after random search: {len(rs_data)}")

## 7. Summary

| Optimizer shape | Example | Pattern |
|---|---|---|
| Ask/tell (update step) | `tpesampler`, custom `Block` subclasses | `(update_step >> data_generator).loop(n)` |
| One-shot (scipy) | `cg`, `lbfgsb`, `nelder_mead` | `one_shot.call(data)` with `maxiter=n` at construction |

Both shapes are returned by `f3dasm.create_optimizer(name, ...)`, or importable directly from `f3dasm.optimization`.

---

**Next:** [Building a Pipeline](../pipeline/pipeline.ipynb)